In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [99]:
key = pd.read_csv('https://raw.githubusercontent.com/HSEAi2025Team-7/keyRateForecast/Alex/cbr.csv')

In [100]:
key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             50 non-null     object 
 1   change_key_rate  50 non-null     object 
 2   basis_points     29 non-null     float64
 3   key_rate         48 non-null     float64
 4   inflation        38 non-null     float64
dtypes: float64(3), object(2)
memory usage: 2.1+ KB


In [101]:
# заполняем пропуски вручную
idx = key[key["inflation"].isna()].index
new_vals = [11, 14.3, 15.9, 17.8, 17.8, 9.5, 6, 5.7, 5.2, 4, 3.1, 3.1]
for i, val in zip(idx, new_vals):
    key.at[i, "inflation"] = val
# заполняем пропуски вручную
idx = key[key["key_rate"].isna()].index
new_vals_k = [20, 20]
for i, val_k in zip(idx, new_vals_k):
    key.at[i, "key_rate"] = val_k
# заполняем медианой basis_point
key['basis_points'] = key['basis_points'].fillna('0')
# добавляем колонку цель по инфляции = 4
key['inflation_target'] = 4
# добавляем колонку разница между инфляцией и целью по инфляции
key['difference'] = key['inflation'] - key['inflation_target']
# добавляем колонку разница межды инфляцией
key['inflation_diff'] = key['inflation'].diff().shift(-1)
# заполняем пропуск медианой
med_inflation_diff = key["inflation_diff"].median()
key["inflation_diff"] = key["inflation_diff"].fillna(med_inflation_diff)
# прошлая ключевая ставка
key['key_rate_last'] = key['key_rate'].shift(-1)
med_key_rate_last = key['key_rate_last'].median()
key['key_rate_last'] = key['key_rate_last'].fillna(med_key_rate_last)
# на сколько изменилась ставка
key['key_rate_last_diff'] = key['key_rate_last'] - key['key_rate_last'].shift(-1)
med_key_rate_diff = key["key_rate_last_diff"].median()
key["key_rate_last_diff"] = key["key_rate_last_diff"].fillna(med_key_rate_diff)

In [7]:
key.isna().sum()

,0
date,0
change_key_rate,0
basis_points,0
key_rate,0
inflation,0
inflation_target,0
difference,0
inflation_diff,0
key_rate_last,0
key_rate_last_diff,0


In [102]:
key.head(50)

,date,change_key_rate,basis_points,key_rate,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff
0,24 октября 2025,снизить,50.0,16.50,8.2,4,4.2,0.0,17.00,-1.00
1,12 сентября 2025,снизить,100.0,17.00,8.2,4,4.2,1.0,18.00,-2.00
2,25 июля 2025,снизить,200.0,18.00,9.2,4,5.2,0.6,20.00,-1.00
3,6 июня 2025,снизить,100.0,20.00,9.8,4,5.8,0.5,21.00,0.00
4,25 апреля 2025,сохранить,0,21.00,10.3,4,6.3,-0.1,21.00,0.00
5,21 марта 2025,сохранить,0,21.00,10.2,4,6.2,-0.2,21.00,0.00
6,14 февраля 2025,сохранить,0,21.00,10.0,4,6.0,-0.5,21.00,0.00
7,20 декабря 2024,сохранить,0,21.00,9.5,4,5.5,-1.1,21.00,2.00
8,25 октября 2024,повысить,200.0,21.00,8.4,4,4.4,0.6,19.00,1.00
9,13 сентября 2024,повысить,100.0,19.00,9.0,4,5.0,-2.0,18.00,2.00


In [79]:
# делим на матрицу признаков и целевую переменную
X1 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y1 = key['change_key_rate']

In [80]:
# делим на матрицу признаков и целевую переменную
X2 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y2 = key['key_rate']

In [ ]:
X1.head()

,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff
0,8.2,4,4.2,0.0,17.0,1.0
1,8.2,4,4.2,1.0,18.0,2.0
2,9.2,4,5.2,0.6,20.0,1.0
3,9.8,4,5.8,0.5,21.0,0.0
4,10.3,4,6.3,-0.1,21.0,0.0


In [ ]:
y1.head()

,change_key_rate
0,снизить
1,снизить
2,снизить
3,снизить
4,сохранить


Рабочая модель

In [95]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# Разделение данных
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=42 , stratify=y1
)

# Масштабирование
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

# Создание модели с фиксированными параметрами
model = LogisticRegression(
    class_weight='balanced',
    penalty= 'l2',
    #solver= 'saga',
    solver= 'liblinear',
)

# Обучение модели
model.fit(X1_train_scaled, y1_train)

# Оценка модели на тесте

y1_pred = model.predict(X1_test_scaled)

In [96]:
print("Accuracy:", accuracy_score(y1_test, y1_pred))
print("f1:", f1_score(y1_test, y1_pred, average='micro'))

Accuracy: 0.5
f1: 0.5


In [97]:
print("Отчёт классификации:\n", classification_report(y1_test, y1_pred,))
print("Матрица ошибок:\n", confusion_matrix(y1_test, y1_pred))

Отчёт классификации:
               precision    recall  f1-score   support

    повысить       0.50      0.67      0.57         3
     снизить       0.67      0.67      0.67         3
   сохранить       0.33      0.25      0.29         4

    accuracy                           0.50        10
   macro avg       0.50      0.53      0.51        10
weighted avg       0.48      0.50      0.49        10

Матрица ошибок:
 [[2 0 1]
 [0 2 1]
 [2 1 1]]


In [98]:
print(y1_pred)
print(y1_test)

['снизить' 'повысить' 'сохранить' 'снизить' 'сохранить' 'повысить'
 'снизить' 'повысить' 'повысить' 'сохранить']
26      снизить
18     повысить
21    сохранить
29      снизить
37     повысить
43    сохранить
23    сохранить
42    сохранить
15     повысить
1       снизить
Name: change_key_rate, dtype: object


In [32]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# your code here
# Разделение данных
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42
    )

    # Масштабирование
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

    # Создание модели с фиксированными параметрами
model = LinearRegression()

                # Обучение модели
model.fit(X2_train_scaled, y2_train)

                # Оценка модели на тесте

y2_pred = model.predict(X2_test_scaled)

In [64]:
mse = mean_squared_error(y2_test, y2_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y2_test, y2_pred)
mape = np.mean(np.abs((y2_test - y2_pred) / y2_test)) * 100  # в процентах
R2 = r2_score(y2_test, y2_pred)
# Вывод результатов
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.6f}%")
print(f"R2:   {R2:.6f}")

MSE:  6.176789
RMSE: 2.485315
MAE:  1.163956
MAPE: 11.091059%
R2:   0.564267


In [65]:
print(y2_pred)
print(y2_test)

[16.22165183  5.94626706 21.61839455  4.99225001 14.52624343  4.74919567
  7.52733464  7.62853316  9.05376772  8.37568443]
13    16.00
39     5.50
30    14.00
45     4.25
17    13.00
48     5.50
26     7.50
25     7.50
32     9.00
19     8.50
Name: key_rate, dtype: float64
